## SRP605513

**paper:** [PMID: 40605499](https://pmc.ncbi.nlm.nih.gov/articles/PMC12328841/) - The Interplay of Ontogeny and Phylogeny at the Transcriptome Level of the Tetrapod Heart, 2025

**date, curator:** 2026-05-01, Sara Carsanaro

**notes**
* dev stage was done manually due to some species having their own dev ontology
* stages from SRA are not exactly the same as what is in the paper (SRA gives ranges while paper is exact) - i used exact stages from figure 1 when SRA range was not possible to annotate on exactly
* for heart annotation - the embryonic heart options in uberon are not great, seems like we use heart for annotation even when embryo
* commented one library with conflicting information for dev stage

### annotation summary

In [20]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,embryonic heart,UBERON:0000948,heart,perfect match


In [21]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,stage 37-38,XAO:1000050,NF stage 37 and 38,perfect match
3,stage 44-45,XAO:1000056,NF stage 44,perfect match
6,stage 47-48,XAO:1000059,NF stage 47,perfect match
9,stage 11,UBERON:0000111,organogenesis stage,other
12,stage 16-17,UBERON:0000111,organogenesis stage,other
15,stage 21-22,UBERON:0007220,late embryonic stage,other
18,stage 10,UBERON:0000111,organogenesis stage,other
21,stage 16,UBERON:0000111,organogenesis stage,other
24,stage 21,UBERON:0007220,late embryonic stage,other
28,HH 18-20,GgalDv:0000030,Hamburger Hamilton stage 18,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP605513"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
  4%|█▊                                          | 2/50 [00:01<00:45,  1.07it/s]curl: (22) The requested URL returned error: 502
>78 ERROR: >78 curl command failed ( Fri May  1 12:52:13 CEST 2026 ) with: 22>78
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi -d db=SRA&id=SRX30667566&rettype=full&retmode=xml&tool=edirect&edirect=16.2&edirect_os=Darwin&email=scarsana%40SORGE42778>78
HTTP/1.0 502 Bad Gateway
nquire -url https://eutils.ncbi.nlm.nih.gov/entrez/eutils/ efetch.fcgi -db SRA -id SRX30667566 -rettype full -retmode xml -tool edirect -edirect 16.2 -edirect_os Darwin -email 

### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX30667555,SRP605513,Illumina HiSeq 2000,SRS26687963,,,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,,,perfect match,,,,8364,,,,,,Zt-chamber-3,SAMN50302213,,,,,,,PMID: 40605499,heart chamber formation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-2-37-38-BC03--Zt,,,,heart chamber formation,,TRANSCRIPTOMIC,cDNA
1,SRX30667554,SRP605513,Illumina HiSeq 2000,SRS26687962,,,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,,,perfect match,,,,8364,,,,,,Zt-chamber-2,SAMN50302212,,,,,,,PMID: 40605499,heart chamber formation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-1-37-38-BC02--Zt,,,,heart chamber formation,,TRANSCRIPTOMIC,cDNA
2,SRX30667553,SRP605513,Illumina HiSeq 2000,SRS26687961,,,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,,,perfect match,,,,8364,,,,,,Zt-chamber-1,SAMN50302211,,,,,,,PMID: 40605499,heart chamber formation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,3-2-37-38-BC01--Zt,,,,heart chamber formation,,TRANSCRIPTOMIC,cDNA
3,SRX30667561,SRP605513,Illumina HiSeq 2000,SRS26687969,,,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,,,perfect match,,,,8364,,,,,,Zt-septation-3,SAMN50302216,,,,,,,PMID: 40605499,heart septation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-2-44-45-BC06--Zt,,,,heart septation,,TRANSCRIPTOMIC,cDNA
4,SRX30667560,SRP605513,Illumina HiSeq 2000,SRS26687968,,,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,,,perfect match,,,,8364,,,,,,Zt-septation-2,SAMN50302215,,,,,,,PMID: 40605499,heart septation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-1-44-45-BC05--Zt,,,,heart septation,,TRANSCRIPTOMIC,cDNA
5,SRX30667559,SRP605513,Illumina HiSeq 2000,SRS26687967,,,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,,,perfect match,,,,8364,,,,,,Zt-septation-1,SAMN50302214,,,,,,,PMID: 40605499,heart septation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,3-2-44-45-BC04--Zt,,,,heart septation,,TRANSCRIPTOMIC,cDNA
6,SRX30667558,SRP605513,Illumina HiSeq 2000,SRS26687966,,,XAO:1000059,NF stage 47,,embryonic heart,stage 47-48,,,perfect match,,,,8364,,,,,,Zt-formed-3,SAMN50302219,,,,,,,PMID: 40605499,fully formed heart,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-2-47-48-BC09--Zt,,,,fully formed heart,,TRANSCRIPTOMIC,cDNA
7,SRX30667557,SRP605513,Illumina HiSeq 2000,SRS26687965,,,XAO:1000059,NF stage 47,,embryonic heart,stage 47-48,,,perfect match,,,,8364,,,,,,Zt-formed-2,SAMN50302218,,,,,,,PMID: 40605499,fully formed heart,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-1-47-48-BC08--Zt

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['embryonic heart']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0000948'
library.loc[:,'anatName'] = 'heart'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'full sampling'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX30667555,SRP605513,Illumina HiSeq 2000,SRS26687963,UBERON:0000948,heart,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,perfect match,full sampling,perfect match,,,,8364,,,,,,Zt-chamber-3,SAMN50302213,,,,,,,PMID: 40605499,heart chamber formation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-2-37-38-BC03--Zt,,,,heart chamber formation,,TRANSCRIPTOMIC,cDNA
1,SRX30667554,SRP605513,Illumina HiSeq 2000,SRS26687962,UBERON:0000948,heart,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,perfect match,full sampling,perfect match,,,,8364,,,,,,Zt-chamber-2,SAMN50302212,,,,,,,PMID: 40605499,heart chamber formation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-1-37-38-BC02--Zt,,,,heart chamber formation,,TRANSCRIPTOMIC,cDNA
2,SRX30667553,SRP605513,Illumina HiSeq 2000,SRS26687961,UBERON:0000948,heart,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,perfect match,full sampling,perfect match,,,,8364,,,,,,Zt-chamber-1,SAMN50302211,,,,,,,PMID: 40605499,heart chamber formation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,3-2-37-38-BC01--Zt,,,,heart chamber formation,,TRANSCRIPTOMIC,cDNA
3,SRX30667561,SRP605513,Illumina HiSeq 2000,SRS26687969,UBERON:0000948,heart,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,perfect match,full sampling,perfect match,,,,8364,,,,,,Zt-septation-3,SAMN50302216,,,,,,,PMID: 40605499,heart septation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-2-44-45-BC06--Zt,,,,heart septation,,TRANSCRIPTOMIC,cDNA
4,SRX30667560,SRP605513,Illumina HiSeq 2000,SRS26687968,UBERON:0000948,heart,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,perfect match,full sampling,perfect match,,,,8364,,,,,,Zt-septation-2,SAMN50302215,,,,,,,PMID: 40605499,heart septation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-1-44-45-BC05--Zt,,,,heart septation,,TRANSCRIPTOMIC,cDNA
5,SRX30667559,SRP605513,Illumina HiSeq 2000,SRS26687967,UBERON:0000948,heart,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,perfect match,full sampling,perfect match,,,,8364,,,,,,Zt-septation-1,SAMN50302214,,,,,,,PMID: 40605499,heart septation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,3-2-44-45-BC04--Zt,,,,heart septation,,TRANSCRIPTOMIC,cDNA
6,SRX30667558,SRP605513,Illumina HiSeq 2000,SRS26687966,UBERON:0000948,heart,XAO:1000059,NF stage 47,,embryonic heart,stage 47-48,perfect match,full sampling,perfect match,,,,8364,,,,,,Zt-formed-3,SAMN50302219,,,,,,,PMID: 40605499,fully formed heart,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-2-47-48-BC09--Zt,,,,fully formed heart,,TRANSCRIPTOMIC,cDNA
7,SRX30667557,SRP605513,Illumina HiSeq 2000,SRS26687965,UBERON:0000948,h

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['E12.5' 'E16.5' 'E9.5' 'HH 18-20' 'HH 25' 'HH 30-32' 'stage 10'
 'stage 11' 'stage 16' 'stage 16-17' 'stage 2' 'stage 21' 'stage 21-22'
 'stage 37-38' 'stage 44-45' 'stage 47-48' 'stage 7']


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [8]:
library.loc[:,'sex'] = 'NA'
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX30667555,SRP605513,Illumina HiSeq 2000,SRS26687963,UBERON:0000948,heart,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,perfect match,full sampling,perfect match,NA,,,8364,,,,,,Zt-chamber-3,SAMN50302213,,,,,,,PMID: 40605499,heart chamber formation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-2-37-38-BC03--Zt,,,,heart chamber formation,,TRANSCRIPTOMIC,cDNA
1,SRX30667554,SRP605513,Illumina HiSeq 2000,SRS26687962,UBERON:0000948,heart,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,perfect match,full sampling,perfect match,NA,,,8364,,,,,,Zt-chamber-2,SAMN50302212,,,,,,,PMID: 40605499,heart chamber formation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-1-37-38-BC02--Zt,,,,heart chamber formation,,TRANSCRIPTOMIC,cDNA
2,SRX30667553,SRP605513,Illumina HiSeq 2000,SRS26687961,UBERON:0000948,heart,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,perfect match,full sampling,perfect match,NA,,,8364,,,,,,Zt-chamber-1,SAMN50302211,,,,,,,PMID: 40605499,heart chamber formation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,3-2-37-38-BC01--Zt,,,,heart chamber formation,,TRANSCRIPTOMIC,cDNA
3,SRX30667561,SRP605513,Illumina HiSeq 2000,SRS26687969,UBERON:0000948,heart,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,perfect match,full sampling,perfect match,NA,,,8364,,,,,,Zt-septation-3,SAMN50302216,,,,,,,PMID: 40605499,heart septation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-2-44-45-BC06--Zt,,,,heart septation,,TRANSCRIPTOMIC,cDNA
4,SRX30667560,SRP605513,Illumina HiSeq 2000,SRS26687968,UBERON:0000948,heart,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,perfect match,full sampling,perfect match,NA,,,8364,,,,,,Zt-septation-2,SAMN50302215,,,,,,,PMID: 40605499,heart septation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-1-44-45-BC05--Zt,,,,heart septation,,TRANSCRIPTOMIC,cDNA
5,SRX30667559,SRP605513,Illumina HiSeq 2000,SRS26687967,UBERON:0000948,heart,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,perfect match,full sampling,perfect match,NA,,,8364,,,,,,Zt-septation-1,SAMN50302214,,,,,,,PMID: 40605499,heart septation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,3-2-44-45-BC04--Zt,,,,heart septation,,TRANSCRIPTOMIC,cDNA
6,SRX30667558,SRP605513,Illumina HiSeq 2000,SRS26687966,UBERON:0000948,heart,XAO:1000059,NF stage 47,,embryonic heart,stage 47-48,perfect match,full sampling,perfect match,NA,,,8364,,,,,,Zt-formed-3,SAMN50302219,,,,,,,PMID: 40605499,fully formed heart,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-2-47-48-BC09--Zt,,,,fully formed heart,,TRANSCRIPTOMIC,cDNA
7,SRX30667557,SRP605513,Illumina HiSeq 2000,SRS26687965,UB

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
#my_protocol = ''
# full_length or 3'
#my_protocolType = ''

#library.loc[:,'protocol'] = my_protocol
#library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX30667555,SRP605513,Illumina HiSeq 2000,SRS26687963,UBERON:0000948,heart,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-chamber-3,SAMN50302213,,,,,,,PMID: 40605499,heart chamber formation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-2-37-38-BC03--Zt,,,,heart chamber formation,,TRANSCRIPTOMIC,cDNA
1,SRX30667554,SRP605513,Illumina HiSeq 2000,SRS26687962,UBERON:0000948,heart,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-chamber-2,SAMN50302212,,,,,,,PMID: 40605499,heart chamber formation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-1-37-38-BC02--Zt,,,,heart chamber formation,,TRANSCRIPTOMIC,cDNA
2,SRX30667553,SRP605513,Illumina HiSeq 2000,SRS26687961,UBERON:0000948,heart,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-chamber-1,SAMN50302211,,,,,,,PMID: 40605499,heart chamber formation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,3-2-37-38-BC01--Zt,,,,heart chamber formation,,TRANSCRIPTOMIC,cDNA
3,SRX30667561,SRP605513,Illumina HiSeq 2000,SRS26687969,UBERON:0000948,heart,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-septation-3,SAMN50302216,,,,,,,PMID: 40605499,heart septation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-2-44-45-BC06--Zt,,,,heart septation,,TRANSCRIPTOMIC,cDNA
4,SRX30667560,SRP605513,Illumina HiSeq 2000,SRS26687968,UBERON:0000948,heart,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-septation-2,SAMN50302215,,,,,,,PMID: 40605499,heart septation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-1-44-45-BC05--Zt,,,,heart septation,,TRANSCRIPTOMIC,cDNA
5,SRX30667559,SRP605513,Illumina HiSeq 2000,SRS26687967,UBERON:0000948,heart,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-septation-1,SAMN50302214,,,,,,,PMID: 40605499,heart septation,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,3-2-44-45-BC04--Zt,,,,heart septation,,TRANSCRIPTOMIC,cDNA
6,SRX30667558,SRP605513,Illumina HiSeq 2000,SRS26687966,UBERON:0000948,heart,XAO:1000059,NF stage 47,,embryonic heart,stage 47-48,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-formed-3,SAMN50302219,,,,,,,PMID: 40605499,fully formed heart,,,01/05/2026,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-2-47-48-BC09--Zt,,,,fully formed heart,,TRANSCRIPTOMIC,cDNA
7,SRX30667557,SRP605513

#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

     #libraryId        SRSId
40  SRX30691973  SRS26687955
42  SRX30667547  SRS26687955


/var/folders/b5/crkp117d43q5mcndnwlrww3w0000gn/T/ipykernel_39895/3311601719.py:43: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  dups = df[duplicateCheck].loc[:,['#libraryId', column]]


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [11]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-05-05'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX30667555,SRP605513,Illumina HiSeq 2000,SRS26687963,UBERON:0000948,heart,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-chamber-3,SAMN50302213,,,,,,,PMID: 40605499,heart chamber formation,,SAC,2026-05-05,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-2-37-38-BC03--Zt,,,,heart chamber formation,,TRANSCRIPTOMIC,cDNA
1,SRX30667554,SRP605513,Illumina HiSeq 2000,SRS26687962,UBERON:0000948,heart,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-chamber-2,SAMN50302212,,,,,,,PMID: 40605499,heart chamber formation,,SAC,2026-05-05,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-1-37-38-BC02--Zt,,,,heart chamber formation,,TRANSCRIPTOMIC,cDNA
2,SRX30667553,SRP605513,Illumina HiSeq 2000,SRS26687961,UBERON:0000948,heart,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-chamber-1,SAMN50302211,,,,,,,PMID: 40605499,heart chamber formation,,SAC,2026-05-05,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,3-2-37-38-BC01--Zt,,,,heart chamber formation,,TRANSCRIPTOMIC,cDNA
3,SRX30667561,SRP605513,Illumina HiSeq 2000,SRS26687969,UBERON:0000948,heart,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-septation-3,SAMN50302216,,,,,,,PMID: 40605499,heart septation,,SAC,2026-05-05,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-2-44-45-BC06--Zt,,,,heart septation,,TRANSCRIPTOMIC,cDNA
4,SRX30667560,SRP605513,Illumina HiSeq 2000,SRS26687968,UBERON:0000948,heart,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-septation-2,SAMN50302215,,,,,,,PMID: 40605499,heart septation,,SAC,2026-05-05,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-1-44-45-BC05--Zt,,,,heart septation,,TRANSCRIPTOMIC,cDNA
5,SRX30667559,SRP605513,Illumina HiSeq 2000,SRS26687967,UBERON:0000948,heart,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-septation-1,SAMN50302214,,,,,,,PMID: 40605499,heart septation,,SAC,2026-05-05,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,3-2-44-45-BC04--Zt,,,,heart septation,,TRANSCRIPTOMIC,cDNA
6,SRX30667558,SRP605513,Illumina HiSeq 2000,SRS26687966,UBERON:0000948,heart,XAO:1000059,NF stage 47,,embryonic heart,stage 47-48,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-formed-3,SAMN50302219,,,,,,,PMID: 40605499,fully formed heart,,SAC,2026-05-05,Total RNA extracted with RNAqueous Micro Kit (AM1931) and subjected to paired-end sequencing (100-bp read length) on the Illumina HiSeq 2000 platform,,5-2-47-48-BC09--Zt,,,,fully formed heart,,TRANSCRIPTOMIC,cDNA
7,

#### comments

In [ ]:
#library.loc[:,'comment'] = ''

#### save complete file with correct columns

In [12]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX30667555,SRP605513,Illumina HiSeq 2000,SRS26687963,UBERON:0000948,heart,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-chamber-3,SAMN50302213,,,,,,,PMID: 40605499,heart chamber formation,,SAC,2026-05-05
1,SRX30667554,SRP605513,Illumina HiSeq 2000,SRS26687962,UBERON:0000948,heart,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-chamber-2,SAMN50302212,,,,,,,PMID: 40605499,heart chamber formation,,SAC,2026-05-05
2,SRX30667553,SRP605513,Illumina HiSeq 2000,SRS26687961,UBERON:0000948,heart,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-chamber-1,SAMN50302211,,,,,,,PMID: 40605499,heart chamber formation,,SAC,2026-05-05
3,SRX30667561,SRP605513,Illumina HiSeq 2000,SRS26687969,UBERON:0000948,heart,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-septation-3,SAMN50302216,,,,,,,PMID: 40605499,heart septation,,SAC,2026-05-05
4,SRX30667560,SRP605513,Illumina HiSeq 2000,SRS26687968,UBERON:0000948,heart,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-septation-2,SAMN50302215,,,,,,,PMID: 40605499,heart septation,,SAC,2026-05-05
5,SRX30667559,SRP605513,Illumina HiSeq 2000,SRS26687967,UBERON:0000948,heart,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-septation-1,SAMN50302214,,,,,,,PMID: 40605499,heart septation,,SAC,2026-05-05
6,SRX30667558,SRP605513,Illumina HiSeq 2000,SRS26687966,UBERON:0000948,heart,XAO:1000059,NF stage 47,,embryonic heart,stage 47-48,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-formed-3,SAMN50302219,,,,,,,PMID: 40605499,fully formed heart,,SAC,2026-05-05
7,SRX30667557,SRP605513,Illumina HiSeq 2000,SRS26687965,UBERON:0000948,heart,XAO:1000059,NF stage 47,,embryonic heart,stage 47-48,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-formed-2,SAMN50302218,,,,,,,PMID: 40605499,fully formed heart,,SAC,2026-05-05
8,SRX30667556,SRP605513,Illumina HiSeq 2000,SRS26687964,UBERON:0000948,heart,XAO:1000059,NF stage 47,,embryonic heart,stage 47-48,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-formed-1,SAMN50302217,,,,,,,PMID: 40605499,fully formed heart,,SAC,2026-05-05
9,SRX30667523,SRP605513,Illumina HiSeq 2000,SRS26687931,UBERON:0000948,heart,UBERON:0000111,organogenesis stage,,embryonic heart,stage 11,perfect match,full sampling,other,NA,,,8479,,,polyA,,,Cp-chamber-1,SAMN50302202,,,,,,,PMID: 40605499,heart chamber formation,,SAC,2026-05-05


### experiment annotations

In [13]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP605513,Transcriptional characterization of heart development across tetrapods,"This project profiled gene expression via RNA-seq in three stages of heart development (chamber formation, septation, and formed heart) in alligator, chicken, frog, mouse, lizard and turtle embryos.",SRA,,,,,,,PRJNA1299454,40605499,,10.1002/jez.b.23312,,


#### experiment and protocol details

In [14]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

50

In [15]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
#experiment.loc[:,'protocol'] = my_protocol
#experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP605513,Transcriptional characterization of heart development across tetrapods,"This project profiled gene expression via RNA-seq in three stages of heart development (chamber formation, septation, and formed heart) in alligator, chicken, frog, mouse, lizard and turtle embryos.",SRA,total,Bgee 1K,50,,,,PRJNA1299454,40605499,,10.1002/jez.b.23312,,


#### paper and xrefs

In [16]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '40605499'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC12328841/'
experiment.loc[:,'DOI'] = '10.1002/jez.b.23312'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP605513,Transcriptional characterization of heart development across tetrapods,"This project profiled gene expression via RNA-seq in three stages of heart development (chamber formation, septation, and formed heart) in alligator, chicken, frog, mouse, lizard and turtle embryos.",SRA,total,Bgee 1K,50,,,,PRJNA1299454,40605499,https://pmc.ncbi.nlm.nih.gov/articles/PMC12328841/,10.1002/jez.b.23312,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [17]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [18]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [19]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [23]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [24]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
63030,ERX2870355,ERP111747,NextSeq 500,ERS2867429,UBERON:0001045,midgut,UBERON:0000066,fully formed stage,,midgut,adult,perfect match,not documented,other,F,Iquitos colony,,43151,SureSelect Strand-Specific RNA Library Prep Kit,full_length,polyA,,,An. darlingi low controls 7dpi rep3,SAMEA5056254,,,,,,,"PMID:41433330, Adult female mosquitoes, starv...",7d post blood feeding (pbf),,ANN,2026-05-05
63031,ERX2870354,ERP111747,NextSeq 500,ERS2867428,UBERON:0001045,midgut,UBERON:0000066,fully formed stage,,midgut,adult,perfect match,not documented,other,F,Iquitos colony,,43151,SureSelect Strand-Specific RNA Library Prep Kit,full_length,polyA,,,An. darlingi low controls 7dpi rep3,SAMEA5056253,,,,,,,"PMID:41433330, Adult female mosquitoes, starv...",7d post blood feeding (pbf),,ANN,2026-05-05
63032,SRX30667555,SRP605513,Illumina HiSeq 2000,SRS26687963,UBERON:0000948,heart,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-chamber-3,SAMN50302213,,,,,,,PMID: 40605499,heart chamber formation,,SAC,2026-05-05
63033,SRX30667554,SRP605513,Illumina HiSeq 2000,SRS26687962,UBERON:0000948,heart,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-chamber-2,SAMN50302212,,,,,,,PMID: 40605499,heart chamber formation,,SAC,2026-05-05
63034,SRX30667553,SRP605513,Illumina HiSeq 2000,SRS26687961,UBERON:0000948,heart,XAO:1000050,NF stage 37 and 38,,embryonic heart,stage 37-38,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-chamber-1,SAMN50302211,,,,,,,PMID: 40605499,heart chamber formation,,SAC,2026-05-05
63035,SRX30667561,SRP605513,Illumina HiSeq 2000,SRS26687969,UBERON:0000948,heart,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-septation-3,SAMN50302216,,,,,,,PMID: 40605499,heart septation,,SAC,2026-05-05
63036,SRX30667560,SRP605513,Illumina HiSeq 2000,SRS26687968,UBERON:0000948,heart,XAO:1000056,NF stage 44,,embryonic heart,stage 44-45,perfect match,full sampling,perfect match,NA,,,8364,,,polyA,,,Zt-septation-2,SAMN50302215,,,,,,,PMID: 40605499,heart septation,,SAC,2026-05-05


In [25]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1223,SRP169074,Oryzias javanicus isolate:RS831 Genome sequenc...,Genome sequence of the euryhaline Javafish med...,SRA,partial,Bgee 1K,11,TruSeq Stranded mRNA,full_length,,PRJNA505405,31988161,https://pmc.ncbi.nlm.nih.gov/articles/PMC7056978/,10.1534/g3.119.400725,,
1224,ERP111747,RNA-Seq of Anopheles darlingi infected or unin...,The availability of the genome sequence of the...,SRA,partial,Bgee 1K,144,SureSelect Strand-Specific RNA Library Prep Kit,full_length,,PRJEB29445,41433330,https://pmc.ncbi.nlm.nih.gov/articles/PMC12758...,10.1371/journal.ppat.1013823,,
1225,SRP605513,Transcriptional characterization of heart deve...,This project profiled gene expression via RNA-...,SRA,total,Bgee 1K,50,,,,PRJNA1299454,40605499,https://pmc.ncbi.nlm.nih.gov/articles/PMC12328...,10.1002/jez.b.23312,,


### add annotations to git

In [26]:
! git pull

Already up to date.


In [27]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [28]:
! git add $git_experiment_path $git_library_path

In [29]:
! git commit -m $commit_message_exp

[develop 757483b] adding annotated bulk experiment SRP605513
 2 files changed, 51 insertions(+)


In [30]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 3.55 KiB | 3.55 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   f56357c..757483b  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push